# 01 · EDA — SQuAD v2 para RoBERTa-base

**Objetivo:** entender el dataset antes de entrenar.

Este notebook:
1. Descarga SQuAD v2 (`rajpurkar/squad_v2`).
2. Cuenta preguntas **contestables** y **no contestables** en train y validation.
3. Mide la longitud de contextos, preguntas y respuestas.
4. Mide la longitud de los contextos **en tokens de RoBERTa** y cuántas ventanas
   (`max_seq_length=384`, `doc_stride=128`) necesita cada ejemplo.
5. Guarda en `logs_roberta/`:
   - `eda_squad2.png` — gráficas
   - `eda_stats.json` — estadísticas

No necesita GPU. Tiempo aproximado: 3–5 minutos.

## 1. Instalación y rutas

In [ ]:
# ⚠️ Es normal que pip muestre un aviso de 'dependency conflicts' con gradio/diffusers:
#    este notebook no usa esos paquetes. Tras esta celda, la primera vez:
#    Entorno de ejecución → Reiniciar sesión, y luego Ejecutar todas de nuevo.
%pip install -q "transformers>=4.46,<5" "datasets>=3.0"

In [ ]:
import os, sys

# ── Rutas ─────────────────────────────────────────────────────────────
# En Colab todo se guarda en Google Drive para que no se pierda si la sesión
# se desconecta. Fuera de Colab se usa la raíz del repositorio.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/roberta_squad2"
else:
    BASE_DIR = os.path.abspath("..")   # el notebook vive en pipeline_roberta/

LOGS_DIR   = os.path.join(BASE_DIR, "logs_roberta")                 # resultados para GitHub
MODEL_DIR  = os.path.join(BASE_DIR, "roberta-base-squad2-final")    # modelo final (NO va a GitHub)
CKPT_DIR   = os.path.join(BASE_DIR, "checkpoints")                  # checkpoints intermedios
PRED_FILE  = os.path.join(BASE_DIR, "predictions_val.json")         # predicciones de validación
os.makedirs(LOGS_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)

## 2. Cargar el dataset

In [ ]:
import json, collections
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "FacebookAI/roberta-base"
MAX_SEQ_LENGTH   = 384
DOC_STRIDE       = 128

ds = load_dataset("rajpurkar/squad_v2")
train, val = ds["train"], ds["validation"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(ds)

In [ ]:
# Un ejemplo contestable y uno no contestable
def show(ex):
    print("Pregunta :", ex["question"])
    print("Contexto :", ex["context"][:300], "...")
    print("Respuesta:", ex["answers"]["text"] if ex["answers"]["text"] else "(sin respuesta)")
    print("-" * 80)

show(next(ex for ex in val if ex["answers"]["text"]))
show(next(ex for ex in val if not ex["answers"]["text"]))

## 3. Contestables vs. no contestables

En SQuAD v2 una pregunta **no contestable** tiene la lista `answers["text"]` vacía.
El modelo debe aprender a devolver una respuesta vacía en esos casos.

In [ ]:
def is_unanswerable(ex):
    return len(ex["answers"]["text"]) == 0

train_unans = [is_unanswerable(ex) for ex in train]
val_unans   = [is_unanswerable(ex) for ex in val]

for name, split, flags in [("train", train, train_unans), ("validation", val, val_unans)]:
    n, u = len(split), sum(flags)
    print(f"{name:<11} total={n:>7,}  contestables={n-u:>7,} ({(n-u)/n:.1%})  no contestables={u:>6,} ({u/n:.1%})")

## 4. Longitudes: contexto, pregunta y respuesta

In [ ]:
# Contexto (caracteres) — por ejemplo, como en el EDA de ModernBERT
ctx_chars = np.array([len(c) for c in train["context"]])

# Contextos únicos en tokens de RoBERTa (muchas preguntas comparten contexto)
unique_ctx = list(dict.fromkeys(train["context"]))
ctx_tokens = np.array([len(ids) for ids in tokenizer(unique_ctx, add_special_tokens=False)["input_ids"]])

# Preguntas (palabras)
q_words = np.array([len(q.split()) for q in train["question"]])
q_words_ans   = q_words[~np.array(train_unans)]
q_words_unans = q_words[np.array(train_unans)]

# Respuestas (todas las respuestas de referencia del train)
answers_flat = [a for ex in train["answers"] for a in ex["text"]]
ans_words = np.array([len(a.split()) for a in answers_flat])
ans_chars = np.array([len(a) for a in answers_flat])

print(f"Contextos únicos: {len(unique_ctx):,}  |  artículos: {len(set(train['title'])):,}")
print(f"Contexto (tokens) media={ctx_tokens.mean():.0f} mediana={np.median(ctx_tokens):.0f} "
      f"p95={np.percentile(ctx_tokens,95):.0f} max={ctx_tokens.max()}")
print(f"Contextos con más de {MAX_SEQ_LENGTH} tokens: {(ctx_tokens > MAX_SEQ_LENGTH).mean():.1%}")
print(f"Pregunta (palabras) media={q_words.mean():.1f}")
print(f"Respuesta (palabras) media={ans_words.mean():.1f} mediana={np.median(ans_words):.0f} max={ans_words.max()}")

## 5. Ventanas por ejemplo (sliding window)

Si pregunta + contexto no caben en 384 tokens, el contexto se divide en varias
ventanas que se solapan 128 tokens. Aquí contamos cuántas ventanas (*features*)
genera cada split: es el número real de muestras que verá el modelo.

In [ ]:
def count_windows(split, chunk=5000):
    counts = []
    for i in range(0, len(split), chunk):
        part = split[i:i + chunk]
        enc = tokenizer([q.strip() for q in part["question"]], part["context"],
                        max_length=MAX_SEQ_LENGTH, truncation="only_second",
                        stride=DOC_STRIDE, return_overflowing_tokens=True)
        counts.append(np.bincount(enc["overflow_to_sample_mapping"], minlength=len(part["id"])))
    return np.concatenate(counts)

train_windows = count_windows(train)
val_windows   = count_windows(val)
for name, w in [("train", train_windows), ("validation", val_windows)]:
    print(f"{name:<11} features={w.sum():>7,}  ejemplos con >1 ventana={(w > 1).mean():.2%}  max ventanas={w.max()}")

## 6. Tipo de pregunta (primera palabra)

In [ ]:
QTYPES = ["what", "who", "how", "when", "which", "where", "why"]
def qtype(q):
    w = q.strip().lower().split()
    return w[0] if w and w[0] in QTYPES else "other"

qt = np.array([qtype(q) for q in train["question"]])
unans_arr = np.array(train_unans)
labels = QTYPES + ["other"]
qt_ans   = [int(((qt == t) & ~unans_arr).sum()) for t in labels]
qt_unans = [int(((qt == t) &  unans_arr).sum()) for t in labels]
for t, a, u in zip(labels, qt_ans, qt_unans):
    print(f"{t:<6} contestables={a:>6,}  no contestables={u:>6,}")

## 7. Gráficas → `eda_squad2.png`

In [ ]:
C_ANS, C_UNANS = "#2E7D32", "#C62828"
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# (a) Contestables vs no contestables por split
ax = axes[0, 0]
splits = ["train", "validation"]
ans_counts   = [len(train) - sum(train_unans), len(val) - sum(val_unans)]
unans_counts = [sum(train_unans), sum(val_unans)]
x = np.arange(2)
ax.bar(x - 0.2, ans_counts, 0.4, label="Contestables", color=C_ANS)
ax.bar(x + 0.2, unans_counts, 0.4, label="No contestables", color=C_UNANS)
for i in range(2):
    tot = ans_counts[i] + unans_counts[i]
    ax.text(x[i] - 0.2, ans_counts[i], f"{ans_counts[i]/tot:.1%}", ha="center", va="bottom", fontsize=9)
    ax.text(x[i] + 0.2, unans_counts[i], f"{unans_counts[i]/tot:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x, splits); ax.set_yscale("log"); ax.set_ylabel("Preguntas (escala log)")
ax.set_title("(a) Contestables vs. no contestables"); ax.legend()

# (b) Longitud del contexto en tokens de RoBERTa
ax = axes[0, 1]
ax.hist(ctx_tokens, bins=60, color="#1565C0", alpha=0.85)
ax.axvline(MAX_SEQ_LENGTH, color="black", ls="--", label=f"max_seq_length = {MAX_SEQ_LENGTH}")
ax.set_title("(b) Longitud de contexto (tokens RoBERTa, contextos únicos)")
ax.set_xlabel("Tokens"); ax.set_ylabel("Contextos"); ax.legend()

# (c) Ventanas por ejemplo
ax = axes[0, 2]
vals, cnts = np.unique(train_windows, return_counts=True)
ax.bar(vals.astype(str), cnts, color="#6A1B9A")
ax.set_yscale("log"); ax.set_title(f"(c) Ventanas por ejemplo (train, stride={DOC_STRIDE})")
ax.set_xlabel("Número de ventanas"); ax.set_ylabel("Ejemplos (escala log)")

# (d) Longitud de pregunta
ax = axes[1, 0]
bins = np.arange(0, 41)
ax.hist(q_words_ans, bins=bins, alpha=0.6, density=True, label="Contestables", color=C_ANS)
ax.hist(q_words_unans, bins=bins, alpha=0.6, density=True, label="No contestables", color=C_UNANS)
ax.set_title("(d) Longitud de pregunta (palabras, train)")
ax.set_xlabel("Palabras"); ax.set_ylabel("Proporción"); ax.legend()

# (e) Longitud de respuesta
ax = axes[1, 1]
ax.hist(np.clip(ans_words, 0, 30), bins=np.arange(1, 32), color="#EF6C00", alpha=0.85)
ax.set_title("(e) Longitud de respuesta (palabras, train; ≥30 agrupado)")
ax.set_xlabel("Palabras"); ax.set_ylabel("Respuestas")

# (f) Tipo de pregunta
ax = axes[1, 2]
x = np.arange(len(labels))
ax.bar(x - 0.2, qt_ans, 0.4, label="Contestables", color=C_ANS)
ax.bar(x + 0.2, qt_unans, 0.4, label="No contestables", color=C_UNANS)
ax.set_xticks(x, labels); ax.set_title("(f) Tipo de pregunta (primera palabra, train)")
ax.set_ylabel("Preguntas"); ax.legend()

fig.suptitle("SQuAD v2 — Análisis exploratorio (tokenizer: FacebookAI/roberta-base)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "eda_squad2.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Guardado:", os.path.join(LOGS_DIR, "eda_squad2.png"))

## 8. Estadísticas → `eda_stats.json`

In [ ]:
def pct(a, b): return round(100 * a / b, 2)

stats = {
    "dataset": "rajpurkar/squad_v2",
    "tokenizer": MODEL_CHECKPOINT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "doc_stride": DOC_STRIDE,
    "train_total": len(train),
    "val_total": len(val),
    "train_unanswerable": int(sum(train_unans)),
    "val_unanswerable": int(sum(val_unans)),
    "train_unanswerable_pct": pct(sum(train_unans), len(train)),
    "val_unanswerable_pct": pct(sum(val_unans), len(val)),
    "train_articles": len(set(train["title"])),
    "train_unique_contexts": len(unique_ctx),
    "context_len_chars_mean": round(float(ctx_chars.mean()), 1),
    "context_len_chars_max": int(ctx_chars.max()),
    "context_len_tokens_mean": round(float(ctx_tokens.mean()), 1),
    "context_len_tokens_median": float(np.median(ctx_tokens)),
    "context_len_tokens_p95": float(np.percentile(ctx_tokens, 95)),
    "context_len_tokens_max": int(ctx_tokens.max()),
    "contexts_longer_than_max_seq_pct": pct((ctx_tokens > MAX_SEQ_LENGTH).sum(), len(ctx_tokens)),
    "train_features": int(train_windows.sum()),
    "val_features": int(val_windows.sum()),
    "train_examples_multi_window_pct": pct((train_windows > 1).sum(), len(train_windows)),
    "val_examples_multi_window_pct": pct((val_windows > 1).sum(), len(val_windows)),
    "question_len_words_mean": round(float(q_words.mean()), 2),
    "answer_len_words_mean": round(float(ans_words.mean()), 2),
    "answer_len_words_median": float(np.median(ans_words)),
    "answer_len_words_max": int(ans_words.max()),
    "answer_len_chars_mean": round(float(ans_chars.mean()), 2),
    "answer_len_chars_max": int(ans_chars.max()),
    "question_types_train": {t: {"answerable": a, "unanswerable": u}
                             for t, a, u in zip(labels, qt_ans, qt_unans)},
}
with open(os.path.join(LOGS_DIR, "eda_stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(json.dumps(stats, indent=2, ensure_ascii=False))
print("✅ Guardado:", os.path.join(LOGS_DIR, "eda_stats.json"))

## Conclusiones para el entrenamiento

Revisa los números de arriba y anota aquí lo que observes. Puntos a mirar:
- ¿Las clases contestable / no contestable están balanceadas en validation? ¿Y en train?
- ¿Qué porcentaje de ejemplos necesita más de una ventana con `max_seq_length=384`?
  Eso justifica usar `doc_stride=128`.
- La mayoría de respuestas son cortas; `max_answer_length=30` tokens en el postproceso
  debería cubrirlas casi todas.

**Siguiente paso:** `02_train_roberta.ipynb`.